In [ ]:
import pandas as pd
import sys, os
os.chdir("../..")   # do korzenia repo
sys.path.insert(0, os.getcwd())
from utils.tools import hash_danych_bezpieczny

INTERIM_PATH = "dane/interim/"

In [2]:
df_fi = pd.read_parquet(f"{INTERIM_PATH}fact_inka.parquet")

In [3]:
hash_wejsciowy = [
    ('fact_inka', ['TowId', 'Data'])
]

for nazwa, kolumny in hash_wejsciowy:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash fact_inka: 012e676339f1c152c164628fdf5fa40ba7751394ae3728f2cbd7dfb54e15de56


#### Do weryfikacji hash **fact_inka:** 012e676339f1c152c164628fdf5fa40ba7751394ae3728f2cbd7dfb54e15de56

In [4]:
# ============================================================
# HARD WYKLUCZENIA
# Kategorie które nie pochodzą z magazynu centralnego SPAR
# lub nie są towarami fizycznymi
# ============================================================

# Prasa i gazety (osobny system zamawiania - kolportaż)
asid_prasa = {132, 133, 273}

# Kiosk - bilety, zapalniczki, leki, baterie (nie z magazynu)
asid_kiosk = {245, 277, 278, 279, 283, 284, 285, 288, 290, 293, 275}

# Doładowania telefoniczne i startery (usługi, nie towary)
asid_doladowania = {47, 48, 255, 290, 330, 331, 332, 334, 335, 337, 338}

# Opakowania i techniczne (nie towary handlowe)
asid_techniczne = {
    256,  # OPAKOWANIA ZWROTNE
    352,  # reklamówki
    247,  # OPAKOWANIA /przemysłowe
    254,  # xx KOSZTY / RABATY
    244,  # Domyślny DO WERYFIKACJI
    354,  # xinne
}

# Łączna lista wykluczeń
lista_hard_wykluczen = (
    asid_prasa |
    asid_kiosk |
    asid_doladowania |
    asid_techniczne
)

print(f"Łącznie do twardego wykluczenia: {len(lista_hard_wykluczen)} AsId")

df_fi_hard = df_fi.copy()

# Filtrowanie AsId
df_fi_hard = df_fi_hard[~df_fi_hard['AsId'].isin(lista_hard_wykluczen)]

# Filtrowanie dokumentów bez NrDok
df_fi_hard = df_fi_hard[df_fi_hard['NrDok'].notna()]

print(f"Przed: {len(df_fi):,} rekordów")
print(f"Po:    {len(df_fi_hard):,} rekordów")
print(f"Usunięto: {len(df_fi) - len(df_fi_hard):,} rekordów")

Łącznie do twardego wykluczenia: 30 AsId
Przed: 4,260,543 rekordów
Po:    4,106,901 rekordów
Usunięto: 153,642 rekordów


In [5]:
df_fi_hard.to_parquet(f"{INTERIM_PATH}fact_inka_hard.parquet", index=False, compression='zstd')
print("Zapisano: dane/interim/fact_inka_hard.parquet")

Zapisano: dane/interim/fact_inka_hard.parquet


In [6]:
hash_wyjsciowy = [
    ('fact_inka_hard', ['TowId', 'Data'])
]

for nazwa, kolumny in hash_wyjsciowy:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash fact_inka_hard: 4b6ce69cbeb2b5a1c6cd820a8857b4aacc16c126807fc5409b946caa8a68ed73


#### Do weryfikacji hash **fact_inka_hard:** 4b6ce69cbeb2b5a1c6cd820a8857b4aacc16c126807fc5409b946caa8a68ed73